In [ ]:
!pip install transformers torchaudio "onnxruntime==1.20.1" "onnx==1.20.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 3.5 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login

hf_token = "YOUR_TOKEN_HERE"

login(token=hf_token)
print("Successfully logged into Hugging Face Hub!")

Successfully logged into Hugging Face Hub!


In [ ]:

from transformers import pipeline

pipe = pipeline("automatic-speech-recognition", model="ai4bharat/indicwav2vec-hindi")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/257 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

[transformers] Could not load the `decoder` for ai4bharat/indicwav2vec-hindi. Defaulting to raw CTC. Error: No module named 'kenlm'
[transformers] Try to install `kenlm`: `pip install kenlm
[transformers] Try to install `pyctcdecode`: `pip install pyctcdecode


In [ ]:
!pip install -q datasets huggingface_hub

from datasets import load_dataset

hindi_valid = load_dataset(
    "ai4bharat/IndicVoices",
    "hindi",
    split="valid",        # the split name inside the config; usually “train” for the data split
    streaming=True       # streams rows on‑demand
)


README.md:   0%|          | 0.00/38.8k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/88 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

In [ ]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 38.8 MB/s eta 0:00:00


In [ ]:
if hasattr(hindi_valid, 'info') and 'valid' in hindi_valid.info.splits:
    total_samples_in_dataset = hindi_valid.info.splits['valid'].num_examples
    print(f"The total number of samples in the 'hindi_valid' dataset (according to dataset metadata) is: {total_samples_in_dataset}")
else:
    print("Could not retrieve the total number of samples from dataset info. This might be a truly streaming dataset without a known total count.")


The total number of samples in the 'hindi_valid' dataset (according to dataset metadata) is: 4740


In [ ]:


num_rows_per_set = total_samples_in_dataset // 2

set1 = hindi_valid.take(num_rows_per_set)

subset_size=num_rows_per_set//2
subset2 = hindi_valid.skip(num_rows_per_set).take(subset_size)
subset3 = hindi_valid.skip(num_rows_per_set+635).take(1735)





Successfully created set1, set2, set3  containing 2370, 1185, 1185 samples respectively.


**Working on Set 1**

In [ ]:
import time
from tqdm.auto import tqdm


predictions_first_half = []
references_first_half = []

print(f"Starting ASR inference on set1 (first {num_rows_per_set} samples)...")


for i, sample in tqdm(enumerate(set1), total=num_rows_per_set, desc="Processing Set1"):
    start_time = time.time() # Start timer for the current sample

    audio_data = sample["audio_filepath"]
    audio_array = audio_data["array"]
    sampling_rate = audio_data["sampling_rate"]
    reference_text = sample["text"]


    transcription = pipe({"sampling_rate": sampling_rate, "raw": audio_array})['text']


    predictions_first_half.append(transcription)
    references_first_half.append(reference_text)

    end_time = time.time() # End timer for the current sample
    iteration_duration = end_time - start_time


    tqdm.write(f"  Sample {i + 1}/{num_rows_per_set} processed in {iteration_duration:.4f} seconds.")

print("Inference for set1 complete.")

Starting ASR inference on set1 (first 2370 samples)...


Processing Set1:   0%|          | 0/2370 [00:00<?, ?it/s]

  Sample 1/2370 processed in 45.3352 seconds.
  Sample 2/2370 processed in 1.7562 seconds.
  Sample 3/2370 processed in 1.8359 seconds.
  Sample 4/2370 processed in 2.8143 seconds.
  Sample 5/2370 processed in 3.4739 seconds.
  Sample 6/2370 processed in 1.2859 seconds.
  Sample 7/2370 processed in 3.1140 seconds.
  Sample 8/2370 processed in 3.8045 seconds.
  Sample 9/2370 processed in 3.7177 seconds.
  Sample 10/2370 processed in 3.7159 seconds.
  Sample 11/2370 processed in 5.3997 seconds.
  Sample 12/2370 processed in 3.4768 seconds.
  Sample 13/2370 processed in 1.0674 seconds.
  Sample 14/2370 processed in 3.1677 seconds.
  Sample 15/2370 processed in 4.1751 seconds.
  Sample 16/2370 processed in 2.9881 seconds.
  Sample 17/2370 processed in 4.7318 seconds.
  Sample 18/2370 processed in 3.6061 seconds.
  Sample 19/2370 processed in 0.7301 seconds.
  Sample 20/2370 processed in 2.5003 seconds.
  Sample 21/2370 processed in 2.5458 seconds.
  Sample 22/2370 processed in 5.7168 secon

In [ ]:
from jiwer import process_words


total_substitutions = 0
total_deletions = 0
total_insertions = 0
total_reference_words = 0

for ref, pred in zip(references_first_half, predictions_first_half):
    metrics = process_words([ref], [pred])
    total_substitutions += metrics.substitutions
    total_deletions += metrics.deletions
    total_insertions+= metrics.insertions

    total_reference_words+= len(ref.split())



print("\n--- Model Results for Set 1---")
print(f"Total reference words: {total_reference_words}")
print(f"Total substitutions: {total_substitutions}")
print(f"Total deletions: {total_deletions}")
print(f"Total insertions: {total_insertions}")



--- Model Results for Set 1---
Total reference words: 37118
Total substitutions: 9919
Total deletions: 3641
Total insertions: 424


**Working on Set 2**

In [ ]:
import time
from tqdm.auto import tqdm


predictions_second_half = []
references_second_half = []

print(f"Starting ASR inference on set2 (second {1735} samples)...")


for i, sample in tqdm(enumerate(subset2), total=1735, desc="Processing Set2"):
    start_time = time.time() # Start timer for the current sample

    audio_data = sample["audio_filepath"]
    audio_array = audio_data["array"]
    sampling_rate = audio_data["sampling_rate"]
    reference_text = sample["text"]


    transcription = pipe({"sampling_rate": sampling_rate, "raw": audio_array})['text']


    predictions_second_half.append(transcription)
    references_second_half.append(reference_text)

    end_time = time.time() # End timer for the current sample
    iteration_duration = end_time - start_time


    tqdm.write(f"  Sample {i + 1}/{1735} processed in {iteration_duration:.4f} seconds.")

print("Inference for set2 complete.")

Starting ASR inference on set2 (second 1735 samples)...


Processing Set2:   0%|          | 0/1735 [00:00<?, ?it/s]

  Sample 1/1735 processed in 2.4960 seconds.
  Sample 2/1735 processed in 0.7078 seconds.
  Sample 3/1735 processed in 6.3470 seconds.
  Sample 4/1735 processed in 0.6108 seconds.
  Sample 5/1735 processed in 0.6600 seconds.
  Sample 6/1735 processed in 0.6128 seconds.
  Sample 7/1735 processed in 4.0525 seconds.
  Sample 8/1735 processed in 1.9267 seconds.
  Sample 9/1735 processed in 0.5226 seconds.
  Sample 10/1735 processed in 0.6541 seconds.
  Sample 11/1735 processed in 1.3199 seconds.
  Sample 12/1735 processed in 1.8886 seconds.
  Sample 13/1735 processed in 0.6429 seconds.
  Sample 14/1735 processed in 0.7725 seconds.
  Sample 15/1735 processed in 1.1325 seconds.
  Sample 16/1735 processed in 0.4715 seconds.
  Sample 17/1735 processed in 10.6743 seconds.
  Sample 18/1735 processed in 7.2384 seconds.
  Sample 19/1735 processed in 12.8772 seconds.
  Sample 20/1735 processed in 5.4719 seconds.
  Sample 21/1735 processed in 10.4908 seconds.
  Sample 22/1735 processed in 7.2970 sec

RuntimeError: decodeAVFrame, /__w/torchcodec/torchcodec/meta-pytorch/torchcodec/src/torchcodec/_core/SingleStreamDecoder.cpp:1397, Could not push packet to decoder: Invalid data found when processing input

In [ ]:
print(len(predictions_first_half))
print(len(references_first_half))

3003
3003


In [ ]:
predictions_first_half[0]

'नमस्ते क्या आप सक्यापित कर सकते हैं कि जो जेनमै चाहता हूँं वह पेटियम माल पर भुरा रंग और नापनापचालिस में उपलब्ध हैं'

In [ ]:
references_first_half[0]

'नमस्ते क्या आप सत्यापित कर सकते हैं कि जो जींस मैं चाहता हूँ वह पेटीएम माल पर भूरे रंग और नाप नाप चालीस में उपलब्ध है'

In [ ]:
from jiwer import process_words,process_characters

#Word level
total_substitutions_words = 0
total_deletions_words = 0
total_insertions_words = 0
total_reference_words = 0

#Character level
total_substitutions_characters = 0
total_deletions_characters = 0
total_insertions_characters = 0
total_reference_characters = 0

for ref, pred in zip(references_first_half, predictions_first_half):
    metrics = process_words([ref], [pred])
    total_substitutions_words += metrics.substitutions
    total_deletions_words += metrics.deletions
    total_insertions_words+= metrics.insertions
    total_reference_words+= len(ref.split())

for ref, pred in zip(references_first_half, predictions_first_half):
    metrics = process_characters([ref], [pred])
    total_substitutions_characters += metrics.substitutions
    total_deletions_characters += metrics.deletions
    total_insertions_characters+= metrics.insertions
    total_reference_characters+= len(ref) # Corrected to count characters



print("\n--- Model Results(Word-level) for Set 1---")
print(f"Total reference words: {total_reference_words}")
print(f"Total substitutions: {total_substitutions_words}")
print(f"Total deletions: {total_deletions_words}")
print(f"Total insertions: {total_insertions_words}")

print("\n--- Model Results(Char-level) for Set 1---")
print(f"Total reference chars: {total_reference_characters}")
print(f"Total substitutions: {total_substitutions_characters}")
print(f"Total deletions: {total_deletions_characters}")
print(f"Total insertions: {total_insertions_characters}")


--- Model Results(Word-level) for Set 1---
Total reference words: 45704
Total substitutions: 12512
Total deletions: 4581
Total insertions: 562

--- Model Results(Char-level) for Set 1---
Total reference chars: 203871
Total substitutions: 15398
Total deletions: 15942
Total insertions: 4374
